**Molecular glue and PROTACs LLM prediction vs. ground truth for the 2nd run**

# Processing Functions

In [ ]:
import requests
import time
from urllib.parse import quote
import pubchempy as pcp


OPSIN_BASE = "https://www.ebi.ac.uk/opsin/ws"
_DASH_CHARS = "‐‑‒–—―−﹣－"
_DASH_TRANS = str.maketrans({c: "-" for c in _DASH_CHARS})


def _normalize_dashes(s):
    if not isinstance(s, str):
        return s
    return s.translate(_DASH_TRANS)


PUBCHEM_BLOCKLIST = {
    "Mei", "Po", "Le",
    "LC-4", "LC-6",
    "D10", "D12", "D15",
    "A13", "13b",
    "SC5", "SC7", "SC10",
    "DBt-10",
    "PS-6",
    "MD9",
    "Cpd 1",
}


def remove_empty_dc50_dmax(df):
    before = len(df)
    cleaned = df.dropna(subset=["DC50", "Dmax"], how="all")
    cleaned = cleaned[~(
        cleaned["DC50"].astype(str).str.strip().isin(["", "nan"])
        & cleaned["Dmax"].astype(str).str.strip().isin(["", "nan"])
    )]
    cleaned = cleaned.reset_index(drop=True)
    after = len(cleaned)
    print(f"Removed {before - after} rows where both DC50 and Dmax are empty")
    print(f"Shape: {before} -> {after}")
    return cleaned


def add_inchikey_opsin(df, sleep_sec=0.1):
    df = df.copy()
    df["Standard_InChIKey"] = pd.NA
    df["Standard_InChIKey_Source"] = pd.NA
    if "SMILES_Source" not in df.columns:
        df["SMILES_Source"] = pd.NA

    has_smiles = df["SMILES"].notna() & (df["SMILES"].astype(str).str.strip() != "")
    df.loc[has_smiles, "SMILES_Source"] = "PAPER"
    print(f"Rows with SMILES from paper: {has_smiles.sum()}")

    success, fail, skip = 0, 0, 0
    failed_rows = []

    for idx, row in df.iterrows():
        iupac = row["IUPAC_Name"]
        if pd.isna(iupac) or str(iupac).strip() == "":
            skip += 1
            continue

        url = f"{OPSIN_BASE}/{quote(str(iupac).strip(), safe='')}.json"
        try:
            resp = requests.get(url, timeout=30)
            if resp.status_code == 200:
                data = resp.json()
                inchikey = data.get("stdinchikey")
                smiles = data.get("smiles")
                if inchikey:
                    df.at[idx, "Standard_InChIKey"] = inchikey
                    df.at[idx, "Standard_InChIKey_Source"] = "OPSIN"
                if not has_smiles[idx] and smiles:
                    df.at[idx, "SMILES"] = smiles
                    df.at[idx, "SMILES_Source"] = "OPSIN"
                success += 1
            else:
                failed_rows.append({"row": idx, "status": resp.status_code, "DOI": row["DOI"], "Compound_Name": row["Compound_Name"], "IUPAC_Name": str(iupac)[:100]})
                fail += 1
        except Exception as e:
            failed_rows.append({"row": idx, "error": str(e), "DOI": row["DOI"], "Compound_Name": row["Compound_Name"], "IUPAC_Name": str(iupac)[:100]})
            fail += 1

        if (idx + 1) % 50 == 0:
            print(f"Processed {idx + 1}/{len(df)} rows...")
        time.sleep(sleep_sec)

    cols = list(df.columns)
    smiles_idx = cols.index("SMILES")
    cols.remove("SMILES_Source")
    cols.insert(smiles_idx + 1, "SMILES_Source")
    df = df[cols]

    print(f"\nDone! Success: {success}, Failed: {fail}, Skipped (no IUPAC): {skip}")
    print(f"Standard_InChIKey filled: {df['Standard_InChIKey'].notna().sum()}")
    print(f"SMILES from PAPER: {(df['SMILES_Source'] == 'PAPER').sum()}")
    print(f"SMILES from OPSIN: {(df['SMILES_Source'] == 'OPSIN').sum()}")
    print(f"SMILES still missing: {df['SMILES'].isna().sum()}")
    if failed_rows:
        print(f"\n--- Failed rows ---")
        for fr in failed_rows:
            print(fr)
    return df


def pubchem_search(df, sleep_sec=0.3, blocklist=None, min_name_length=3):
    """Fill Standard_InChIKey (and IUPAC_Name / SMILES if missing) via PubChem.
    """
    df = df.copy()
    if blocklist is None:
        blocklist = PUBCHEM_BLOCKLIST

    def has_value(v):
        return pd.notna(v) and str(v).strip() != "" and str(v).strip().lower() != "nan"

    def is_numeric_name(s):
        return s.replace(".", "").replace("-", "").isdigit()

    def cname_usable(c):
        if not c:
            return False
        if c in blocklist or _normalize_dashes(c) in blocklist:
            return False
        if is_numeric_name(c):
            return False
        if len(c) < min_name_length:
            return False
        return True

    def search_name_with_synonym(raw_query):
        query = _normalize_dashes(raw_query).strip()
        if not query:
            return None, "empty after normalization"
        try:
            compounds = pcp.get_compounds(query, "name")
            if not compounds:
                return None, "0 results"
            top = compounds[0]
            syn_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{top.cid}/synonyms/JSON"
            syn_resp = requests.get(syn_url, timeout=30)
            if syn_resp.status_code != 200:
                return None, f"synonyms HTTP {syn_resp.status_code} (CID={top.cid})"
            syns = syn_resp.json()["InformationList"]["Information"][0]["Synonym"]
            q_cmp = query.lower()
            if not any(_normalize_dashes(s).lower() == q_cmp for s in syns):
                return None, f"not in synonyms (CID={top.cid})"
            return {"cid": top.cid, "iupac_name": top.iupac_name,
                    "smiles": top.smiles, "inchikey": top.inchikey}, "ok"
        except Exception as e:
            return None, f"error: {e}"

    def search_smiles(query):
        try:
            compounds = pcp.get_compounds(query, "smiles")
            if not compounds:
                return None, "0 results"
            top = compounds[0]
            return {"cid": top.cid, "iupac_name": top.iupac_name,
                    "smiles": top.smiles, "inchikey": top.inchikey}, "ok"
        except Exception as e:
            return None, f"error: {e}"

    def chem_search(iupac, smiles):
        if iupac:
            r, reason = search_name_with_synonym(iupac)
            time.sleep(sleep_sec)
            if r:
                return r, "IUPAC", reason
            iupac_reason = reason
        else:
            iupac_reason = "no IUPAC"
        if smiles:
            r, reason = search_smiles(smiles)
            time.sleep(sleep_sec)
            if r:
                return r, "SMILES", reason
            smiles_reason = reason
        else:
            smiles_reason = "no SMILES"
        return None, None, f"IUPAC={iupac_reason}, SMILES={smiles_reason}"

    for col in ("IUPAC_Name_Source", "Standard_InChIKey",
                "Standard_InChIKey_Source", "SMILES_Source"):
        if col not in df.columns:
            df[col] = pd.NA

    needs_key = df["Standard_InChIKey"].isna() | (df["Standard_InChIKey"].astype(str).str.strip() == "")
    to_search = df[needs_key].index

    queries = {}
    for idx in to_search:
        cname = str(df.at[idx, "Compound_Name"]).strip() if has_value(df.at[idx, "Compound_Name"]) else ""
        iupac = str(df.at[idx, "IUPAC_Name"]).strip()    if has_value(df.at[idx, "IUPAC_Name"])    else ""
        smi   = str(df.at[idx, "SMILES"]).strip()        if has_value(df.at[idx, "SMILES"])        else ""
        queries.setdefault((cname, iupac, smi), []).append(idx)

    print(f"Rows missing Standard_InChIKey: {len(to_search)} ({len(queries)} unique (cname, iupac, smiles) triples)")

    def short(s, n=40):
        return (s[:n] + "...") if s and len(s) > n else s

    cache = {}
    for (cname, iupac, smi), idxs in queries.items():
        usable_c = cname_usable(cname)
        has_chem = bool(iupac) or bool(smi)
        cs = short(cname) if cname else "(no cname)"

        result, method, reason = None, None, ""

        if usable_c and not has_chem:
            r, reason_c = search_name_with_synonym(cname)
            time.sleep(sleep_sec)
            if r:
                result, method, reason = r, "cname", reason_c
            else:
                reason = f"cname={reason_c}"
        elif usable_c and has_chem:
            r, reason_c = search_name_with_synonym(cname)
            time.sleep(sleep_sec)
            if r:
                result, method, reason = r, "cname", reason_c
            else:
                result, method, reason_chem = chem_search(iupac, smi)
                reason = f"cname={reason_c}; {reason_chem}" if not result else f"after cname miss: {reason_chem}"
        elif not usable_c and has_chem:
            result, method, reason_chem = chem_search(iupac, smi)
            reason = f"cname unusable; {reason_chem}"
        else:
            reason = "skip: no usable name and no IUPAC/SMILES"

        cache[(cname, iupac, smi)] = result

        if result:
            print(f"  MATCH '{cs}' via {method}: CID={result['cid']}, InChIKey={result['inchikey']}")
        else:
            print(f"  MISS  '{cs}': {reason}")

    filled_key = filled_iupac = filled_smi = 0
    for (cname, iupac, smi), idxs in queries.items():
        result = cache[(cname, iupac, smi)]
        if result is None:
            continue
        for idx in idxs:
            if result["inchikey"]:
                df.at[idx, "Standard_InChIKey"] = result["inchikey"]
                df.at[idx, "Standard_InChIKey_Source"] = "PUBCHEM"
                filled_key += 1
            if not has_value(df.at[idx, "IUPAC_Name"]) and result.get("iupac_name"):
                df.at[idx, "IUPAC_Name"] = result["iupac_name"]
                df.at[idx, "IUPAC_Name_Source"] = "PUBCHEM"
                filled_iupac += 1
            if not has_value(df.at[idx, "SMILES"]) and result.get("smiles"):
                df.at[idx, "SMILES"] = result["smiles"]
                df.at[idx, "SMILES_Source"] = "PUBCHEM"
                filled_smi += 1

    matched = sum(1 for v in cache.values() if v is not None)
    print(f"\n--- PubChem Results ---")
    print(f"Matched triples: {matched}/{len(queries)}")
    print(f"Standard_InChIKey filled: {filled_key}")
    print(f"IUPAC_Name filled: {filled_iupac}")
    print(f"SMILES filled: {filled_smi}")
    print(f"Standard_InChIKey still missing: {(df['Standard_InChIKey'].isna() | (df['Standard_InChIKey'].astype(str).str.strip() == '')).sum()}")
    return df

def add_connectivity_key(df):
    if 'Standard_InChIKey' in df.columns:
        df['Connectivity_Key'] = df['Standard_InChIKey'].str[:14]
    return df

# Porcess PROTACs Extracted Data

In [1]:
import pandas as pd
import os
import glob


base_dir = "/Users/yaochenr/project/molecular_glue_extractor/output/results"
batches = {
    "batch_1": "260501_1226",
    "batch_2": "260501_2158",
    "batch_3": "260501_2234",
    "batch_4": "260501_2302",
    "batch_5": "260501_2332",
    "batch_6": "260501_2358",
    "batch_7": "260502_0031",
    "batch_8": "260502_0057",
    "batch_9": "260502_0117",
    "batch_10": "260502_0142",
    "batch_11": "260502_0203",
    "batch_12": "260502_0227",
    "batch_13": "260502_0253",
    "batch_14": "260502_0312",
}

dfs = []

for batch_name, folder in batches.items():
    path = os.path.join(base_dir, folder, "data_processed.csv")
    df_protacs = pd.read_csv(path)
    dfs.append(df_protacs)
    print(f"{batch_name} ({folder}): {len(df_protacs)} rows")

merged_protacs = pd.concat(dfs, ignore_index=True)
if "source" in merged_protacs.columns:
    merged_protacs = merged_protacs.drop(columns=["source"])
print(f"\nTotal rows: {len(merged_protacs)}")
print(f"Unique DOIs: {merged_protacs['DOI'].nunique()}")
print(f"Unknown DOIs: {(merged_protacs['DOI'] == 'Unknown').sum()}")
print(f"Columns: {list(merged_protacs.columns)}")
merged_protacs.head()

batch_1 (260501_1226): 38 rows
batch_2 (260501_2158): 45 rows
batch_3 (260501_2234): 35 rows
batch_4 (260501_2302): 142 rows
batch_5 (260501_2332): 110 rows
batch_6 (260501_2358): 48 rows
batch_7 (260502_0031): 46 rows
batch_8 (260502_0057): 21 rows
batch_9 (260502_0117): 65 rows
batch_10 (260502_0142): 49 rows
batch_11 (260502_0203): 38 rows
batch_12 (260502_0227): 33 rows
batch_13 (260502_0253): 45 rows
batch_14 (260502_0312): 70 rows

Total rows: 785
Unique DOIs: 118
Unknown DOIs: 0
Columns: ['DOI', 'Compound_Name', 'IUPAC_Name', 'SMILES', 'Degradation_Target', 'Recruiter', 'Assay', 'Cell_Line', 'DC50', 'DC50_units', 'DC50_h', 'Dmax', 'Dmax_h', 'Dmax_conc']


,DOI,Compound_Name,IUPAC_Name,SMILES,Degradation_Target,Recruiter,Assay,Cell_Line,DC50,DC50_units,DC50_h,Dmax,Dmax_h,Dmax_conc
0,10.1038/s42003-020-0868-6,1,NaN,NaN,RIPK2,VHL,capillary-based immunoassay,THP-1,2.0,nM,18 h,NaN,NaN,NaN
1,10.1038/s42003-020-0868-6,2,(S)-7-(2-(2-(2-(2-((4-(benzo[d]thiazol-5-ylami...,NaN,RIPK2,IAP,capillary-based immunoassay,THP-1,0.40,nM,18 h,NaN,NaN,NaN
2,10.1038/s42003-020-0868-6,3,(S)-14-((4-(Benzo[d]thiazol-5-ylamino)-6-(tert...,NaN,RIPK2,cereblon,capillary-based immunoassay,THP-1,2.5,nM,18 h,NaN,NaN,NaN
3,10.1038/s42003-020-0868-6,4,(S)-N-((S)-1-(1-(2-(4-(2-((6-(tert-Butylsulfon...,NaN,RIPK2,IAP,Simple Western (capillary-based) immunoassay,human PBMCs,12.6,nM,24 h,69.2,24 h,NaN
4,10.1038/s42003-020-0868-6,6,"5-(4-(3-((6-(tert-Butylsulfonyl)-4-((4,5-dimet...",NaN,RIPK2,IAP,Simple Western (capillary-based) immunoassay,human PBMCs,0.40,nM,24 h,94.3,24 h,NaN


In [3]:
data_dir = "/Users/yaochenr/project/molecular_glue_extractor/output/results/260502_protac_data_all_reps/data_2nd"
os.makedirs(data_dir, exist_ok=True)
output_path = os.path.join(data_dir, "260502_merged_protac_data_processed_2nd.csv")
merged_protacs.to_csv(output_path, index=False)
print(f"Saved merged data to: {output_path}")
print(f"Shape: {merged_protacs.shape}")

Saved merged data to: /Users/yaochenr/project/molecular_glue_extractor/output/results/260502_protac_data_all_reps/data_2nd/260502_merged_protac_data_processed_2nd.csv
Shape: (785, 14)


### Remove rows where DC50 & Dmax are both empty

In [5]:
merged_cleaned_protac = remove_empty_dc50_dmax(merged_protacs)

Removed 117 rows where both DC50 and Dmax are empty
Shape: 785 -> 668


In [ ]:
output_merged = os.path.join(data_dir, "260502_merged_protac_data_processed_cleaned_2nd.csv")
merged_cleaned_protac.to_csv(output_merged, index=False)
print(f"Saved cleaned merged data to: {output_merged}")
print(f"Shape: {merged_cleaned_protac.shape}")

Saved cleaned merged data to: /Users/yaochenr/project/molecular_glue_extractor/output/results/260502_protac_data_all_reps/data_2nd/260502_merged_protac_data_processed_cleaned_2nd.csv
Shape: (668, 14)


### Add Standard InChIKey via OPSIN

In [7]:
merged_cleaned_protac = add_inchikey_opsin(merged_cleaned_protac)

Rows with SMILES from paper: 102
Processed 50/668 rows...
Processed 100/668 rows...
Processed 150/668 rows...
Processed 250/668 rows...
Processed 300/668 rows...
Processed 350/668 rows...
Processed 400/668 rows...
Processed 450/668 rows...
Processed 500/668 rows...
Processed 550/668 rows...
Processed 650/668 rows...

Done! Success: 383, Failed: 96, Skipped (no IUPAC): 189
Standard_InChIKey filled: 383
SMILES from PAPER: 102
SMILES from OPSIN: 302
SMILES still missing: 264

--- Failed rows ---
{'row': 1, 'status': 404, 'DOI': '10.1038/s42003-020-0868-6', 'Compound_Name': '2', 'IUPAC_Name': '(S)-7-(2-(2-(2-(2-((4-(benzo[d]thiazol-5-ylamino)-6-(tert-butylsulfonyl)quinolin-7-yl)oxy)ethoxy)eth'}
{'row': 50, 'status': 404, 'DOI': '10.1080/14756366.2022.2062338', 'Compound_Name': '16', 'IUPAC_Name': 'N-(2-(2, 6-dioxopiperidin-3-yl)-1,3-dioxoisoindolin-4-yl)-2-(2-(2-(2-(4-methyl-3-oxo-3, 4-dihydroqui'}
{'row': 88, 'status': 404, 'DOI': '10.1021/acs.jmedchem.2c00771', 'Compound_Name': 'NU223612

In [8]:
output_path = os.path.join(data_dir, "260502_merged_protac_data_processed_cleaned_inchikey_2nd.csv")
merged_cleaned_protac.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")
print(f"Shape: {merged_cleaned_protac.shape}")

Saved to: /Users/yaochenr/project/molecular_glue_extractor/output/results/260502_protac_data_all_reps/data_2nd/260502_merged_protac_data_processed_cleaned_inchikey_2nd.csv
Shape: (668, 17)


### PubChem search

In [9]:
merged_cleaned_protac = pubchem_search(merged_cleaned_protac)

Rows missing Standard_InChIKey: 285 (130 unique (cname, iupac, smiles) triples)
  MISS  '1': skip: no usable name and no IUPAC/SMILES
  MISS  '2': cname unusable; IUPAC=0 results, SMILES=no SMILES
  MATCH 'KY02111' via cname: CID=8582409, InChIKey=LXFKEVQQSKQXPR-UHFFFAOYSA-N
  MATCH 'UBX-382' via cname: CID=166545675, InChIKey=WIAKHDZSMZDZQL-UHFFFAOYSA-N
  MISS  '34 (CST651)': cname=0 results
  MISS  '16': cname unusable; IUPAC=0 results, SMILES=no SMILES
  MISS  '2c': skip: no usable name and no IUPAC/SMILES
  MISS  'PRO-6 E': cname=0 results
  MATCH 'LC-3' via cname: CID=156885419, InChIKey=QSJKPPVXARILLJ-GYUTUTRISA-N
  MISS  'LC-4': skip: no usable name and no IUPAC/SMILES
  MATCH 'LC-5' via cname: CID=159353560, InChIKey=LHQBPRGJSOSPMT-JCKBJUPWSA-N
  MISS  'LC-6': skip: no usable name and no IUPAC/SMILES
  MISS  '5': skip: no usable name and no IUPAC/SMILES
  MATCH 'NU223612' via cname: CID=166176927, InChIKey=PLYOHMQINGIYDN-LVMIGYEDSA-N
  MISS  '14': skip: no usable name and no IU

In [10]:
output_path = os.path.join(data_dir, "260502_merged_protac_data_processed_cleaned_inchikey_pubchem_2nd.csv")
merged_cleaned_protac.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")
print(f"Shape: {merged_cleaned_protac.shape}")

Saved to: /Users/yaochenr/project/molecular_glue_extractor/output/results/260502_protac_data_all_reps/data_2nd/260502_merged_protac_data_processed_cleaned_inchikey_pubchem_2nd.csv
Shape: (668, 18)


### Add connectivity_key

In [11]:
merged_cleaned_protac = add_connectivity_key(merged_cleaned_protac)

In [12]:
merged_cleaned_protac

,DOI,Compound_Name,IUPAC_Name,SMILES,SMILES_Source,Degradation_Target,Recruiter,Assay,Cell_Line,DC50,DC50_units,DC50_h,Dmax,Dmax_h,Dmax_conc,Standard_InChIKey,Standard_InChIKey_Source,IUPAC_Name_Source,Connectivity_Key
0,10.1038/s42003-020-0868-6,1,NaN,NaN,<NA>,RIPK2,VHL,capillary-based immunoassay,THP-1,2.0,nM,18 h,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>
1,10.1038/s42003-020-0868-6,2,(S)-7-(2-(2-(2-(2-((4-(benzo[d]thiazol-5-ylami...,NaN,<NA>,RIPK2,IAP,capillary-based immunoassay,THP-1,0.40,nM,18 h,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>
2,10.1038/s42003-020-0868-6,3,(S)-14-((4-(Benzo[d]thiazol-5-ylamino)-6-(tert...,S1C=NC2=C1C=CC(=C2)NC2=CC=NC1=CC(=C(C=C21)S(=O...,OPSIN,RIPK2,cereblon,capillary-based immunoassay,THP-1,2.5,nM,18 h,NaN,NaN,NaN,BRZGXSNAXRPPFI-DHUJRADRSA-N,OPSIN,<NA>,BRZGXSNAXRPPFI
3,10.1038/s42003-020-0868-6,4,(S)-N-((S)-1-(1-(2-(4-(2-((6-(tert-Butylsulfon...,C(C)(C)(C)S(=O)(=O)C=1C=C2C(=NC=NC2=CC1OCCN1CC...,OPSIN,RIPK2,IAP,Simple Western (capillary-based) immunoassay,human PBMCs,12.6,nM,24 h,69.2,24 h,NaN,HZMQXNZSFFXKMT-LCPJFFRWSA-N,OPSIN,<NA>,HZMQXNZSFFXKMT
4,10.1038/s42003-020-0868-6,6,"5-(4-(3-((6-(tert-Butylsulfonyl)-4-((4,5-dimet...",C(C)(C)(C)S(=O)(=O)C=1C=C2C(=NC=NC2=CC1OCCCN1C...,OPSIN,RIPK2,IAP,Simple Western (capillary-based) immunoassay,human PBMCs,0.40,nM,24 h,94.3,24 h,NaN,CHNBEZLJAQUZEX-LFRONEFISA-N,OPSIN,<NA>,CHNBEZLJAQUZEX
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
663,10.1101/gad.349717.122,BTR2004,NaN,NaN,<NA>,BRD3,KLHL20,Western blot densitometry,PC3,~87,nM,4,100,4,10 μM,<NA>,<NA>,<NA>,<NA>
664,10.1101/gad.349717.122,BTR2004,NaN,NaN,<NA>,BRD4,KLHL20,Western blot densitometry,PC3,~777,nM,4,100,4,10 μM,<NA>,<NA>,<NA>,<NA>
665,10.1039/d3cb00103b,FS-ARV-825,NaN,NaN,<NA>,GSPT1,CRBN,GFP/mCherry high-content imaging (Operetta),Flp293T cells stably expressing GFP-GSPT1domai...,~500,nM,5 h,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>
666,10.1039/d3cb00103b,ARV-825,"2-[(9S)-7-(4-chlorophenyl)-4,5,13-trimethyl-3-...",CC1=C(SC2=C1C(=N[C@H](C3=NN=C(N32)C)CC(=O)NC4=...,PUBCHEM,IKZF1,CRBN,HiBiT lytic luminescence assay,MOLT4 HiBiT-IKZF1 cells,26 ± 2,nM,24 h,~95,24 h,NaN,RWLOGRLTDKDANT-TYIYNAFKSA-N,PUBCHEM,PUBCHEM,RWLOGRLTDKDANT


In [13]:
output_path = os.path.join(data_dir, "260502_merged_protac_data_processed_cleaned_inchikey_pubchem_ck_2nd.csv")
merged_cleaned_protac.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")
print(f"Shape: {merged_cleaned_protac.shape}")

Saved to: /Users/yaochenr/project/molecular_glue_extractor/output/results/260502_protac_data_all_reps/data_2nd/260502_merged_protac_data_processed_cleaned_inchikey_pubchem_ck_2nd.csv
Shape: (668, 19)


# Porcess Molecular Glues Extracted Data

In [2]:
import pandas as pd
import os

glue_path = '/Users/yaochenr/project/molecular_glue_extractor/output/results/260504_1500/data_processed.csv'
merged_glues = pd.read_csv(glue_path)
merged_glues.head()

,DOI,source,Compound_Name,IUPAC_Name,SMILES,Degradation_Target,Recruiter,Assay,Cell_Line,DC50,DC50_units,DC50_h,Dmax,Dmax_h,Dmax_conc
0,10.1038/s41467-022-28907-3,main_text,indisulam,NaN,NaN,RBM39,DCAF15,LC–MS-based global label-free proteomics,IMR-32,NaN,NaN,NaN,NaN,NaN,NaN
1,10.1038/s41467-022-28907-3,main_text,indisulam,NaN,NaN,RBM39,DCAF15,Western blot,IMR-32,NaN,NaN,NaN,NaN,NaN,NaN
2,10.1038/s41467-022-28907-3,main_text,indisulam,NaN,NaN,RBM39,DCAF15,Western blot,KELLY,NaN,NaN,NaN,NaN,NaN,NaN
3,10.1038/s41467-022-28907-3,main_text,E7820,NaN,NaN,RBM39,DCAF15,Western blot,KELLY,NaN,NaN,NaN,NaN,NaN,NaN
4,10.1002/mabi.202400427,main_text,PRO-Lin7,NaN,NaN,Lin28,CRBN,Western blot,PA-1,≈ 300,nM,NaN,75,NaN,1 µM


### Remove rows where DC50 & Dmax are both empty

In [3]:
merged_cleaned_glues = remove_empty_dc50_dmax(merged_glues)

Removed 183 rows where both DC50 and Dmax are empty
Shape: 424 -> 241


In [4]:
data_dir_glues = "/Users/yaochenr/project/molecular_glue_extractor/output/results/260504-glue-data-all-rep/data_2nd"
os.makedirs(data_dir_glues, exist_ok=True)
output_merged = os.path.join(data_dir_glues, "260504_merged_glue_data_processed_cleaned_2nd.csv")
merged_cleaned_glues.to_csv(output_merged, index=False)
print(f"Saved cleaned merged data to: {output_merged}")
print(f"Shape: {merged_cleaned_glues.shape}")

Saved cleaned merged data to: /Users/yaochenr/project/molecular_glue_extractor/output/results/260504-glue-data-all-rep/data_2nd/260504_merged_glue_data_processed_cleaned_2nd.csv
Shape: (241, 15)


### Add Standard InChIKey via OPSIN

In [5]:
merged_cleaned_glues = add_inchikey_opsin(merged_cleaned_glues)

Rows with SMILES from paper: 32
Processed 50/241 rows...
Processed 200/241 rows...

Done! Success: 135, Failed: 46, Skipped (no IUPAC): 60
Standard_InChIKey filled: 135
SMILES from PAPER: 32
SMILES from OPSIN: 105
SMILES still missing: 104

--- Failed rows ---
{'row': 59, 'status': 404, 'DOI': '10.1038/s41467-025-58431-z', 'Compound_Name': '13', 'IUPAC_Name': "3-(1'-(cyclohexylmethyl)-6-oxo-6,8-dihydro-2H,7H-spiro[furo[2,3 e]isoindole-3,4'-piperidin]-7-yl)pip"}
{'row': 60, 'status': 404, 'DOI': '10.1038/s41467-025-58431-z', 'Compound_Name': '14', 'IUPAC_Name': "3-(6-oxo-1'-(pyridin-2-ylmethyl)-6,8-dihydro-2H,7H-spiro[furo[2,3 e]isoindole-3,4'-piperidin]-7-yl)p"}
{'row': 61, 'status': 404, 'DOI': '10.1038/s41467-025-58431-z', 'Compound_Name': '15', 'IUPAC_Name': "3-(6-oxo-1'-(thiophen-3-ylmethyl)-6,8-dihydro-2H,7H-spiro[furo[2,3 e]isoindole-3,4'-piperidin]-7-yl)"}
{'row': 63, 'status': 404, 'DOI': '10.1038/s41467-025-58431-z', 'Compound_Name': '7', 'IUPAC_Name': "3-(6'-oxo-4-phenoxy-6',

In [6]:
output_path = os.path.join(data_dir_glues, "260504_merged_glue_data_processed_cleaned_inchikey_2nd.csv")
merged_cleaned_glues.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")
print(f"Shape: {merged_cleaned_glues.shape}")

Saved to: /Users/yaochenr/project/molecular_glue_extractor/output/results/260504-glue-data-all-rep/data_2nd/260504_merged_glue_data_processed_cleaned_inchikey_2nd.csv
Shape: (241, 18)


### PubChem Search

In [7]:
merged_cleaned_glues = pubchem_search(merged_cleaned_glues)

Rows missing Standard_InChIKey: 106 (78 unique (cname, iupac, smiles) triples)
  MISS  'PRO-Lin7': cname=0 results
  MISS  'MG-Lin1': cname=0 results
  MATCH 'lenalidomide' via cname: CID=216326, InChIKey=GOTYRUGSSMKFNF-UHFFFAOYSA-N
  MATCH 'dBET57' via cname: CID=118912822, InChIKey=CZRLOIDJCMKJHE-UXMRNZNESA-N
  MATCH 'A6' via SMILES: CID=166642468, InChIKey=FQLCLMHMOFLVSV-UHFFFAOYSA-N
  MISS  '13-7': skip: no usable name and no IUPAC/SMILES
  MATCH 'DKY709' via cname: CID=137519326, InChIKey=OMISHRJQMYQPMG-UHFFFAOYSA-N
  MISS  '13': cname unusable; IUPAC=0 results, SMILES=no SMILES
  MISS  '14': cname unusable; IUPAC=0 results, SMILES=no SMILES
  MISS  '15': cname unusable; IUPAC=0 results, SMILES=no SMILES
  MISS  '7': cname unusable; IUPAC=0 results, SMILES=no SMILES
  MISS  '9': cname unusable; IUPAC=0 results, SMILES=no SMILES
  MATCH 'pomalidomide' via cname: CID=134780, InChIKey=UVSMNLNDYGZFPF-UHFFFAOYSA-N
  MISS  'iDeg-6': cname=0 results
  MATCH 'MGD-28' via cname: CID=172871

In [8]:
output_path = os.path.join(data_dir_glues, "260504_merged_glue_data_processed_cleaned_inchikey_pubchem_2nd.csv")
merged_cleaned_glues.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")
print(f"Shape: {merged_cleaned_glues.shape}")

Saved to: /Users/yaochenr/project/molecular_glue_extractor/output/results/260504-glue-data-all-rep/data_2nd/260504_merged_glue_data_processed_cleaned_inchikey_pubchem_2nd.csv
Shape: (241, 19)


### Add connectivity_key

In [9]:
merged_cleaned_glues = add_connectivity_key(merged_cleaned_glues)

In [10]:
merged_cleaned_glues[['Compound_Name', 'Standard_InChIKey', 'Connectivity_Key']]

,Compound_Name,Standard_InChIKey,Connectivity_Key
0,PRO-Lin7,<NA>,<NA>
1,MG-Lin1,<NA>,<NA>
2,5a,MSHTVRYXGFREAN-UHFFFAOYSA-N,MSHTVRYXGFREAN
3,7d,QRQMHYISDDHZBY-UHFFFAOYSA-N,QRQMHYISDDHZBY
4,7f,NSEUGMATEWSNTO-UHFFFAOYSA-N,NSEUGMATEWSNTO
...,...,...,...
236,CCT373566,GSGDUDAFETZSPM-IATAILRESA-N,GSGDUDAFETZSPM
237,CCT373566,GSGDUDAFETZSPM-IATAILRESA-N,GSGDUDAFETZSPM
238,CCT373566,GSGDUDAFETZSPM-IATAILRESA-N,GSGDUDAFETZSPM
239,CCT373567,GSGDUDAFETZSPM-DHZVRSILSA-N,GSGDUDAFETZSPM


In [11]:
output_path = os.path.join(data_dir_glues, "260504_merged_glue_data_processed_cleaned_inchikey_pubchem_ck_2nd.csv")
merged_cleaned_glues.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")
print(f"Shape: {merged_cleaned_glues.shape}")

Saved to: /Users/yaochenr/project/molecular_glue_extractor/output/results/260504-glue-data-all-rep/data_2nd/260504_merged_glue_data_processed_cleaned_inchikey_pubchem_ck_2nd.csv
Shape: (241, 20)
